# DeepLense Test 1 - Multi-Class Lens Classification

This notebook presents a PyTorch pipeline for classifying single-channel galaxy images into the three DeepLense Test 1 categories: `no`, `sphere`, and `vort`. It is a notebook-structured version of the current `train.py` configuration, keeping the model and evaluation logic unchanged while reorganizing the workflow into clearly separated sections.

## Notebook Goal
- Train a three-class classifier on `dataset/train`
- Reserve a stratified internal validation split for model selection
- Report standard validation metrics together with ROC/AUC analysis
- Add test-time augmentation as an inference-only enhancement

## Current Configuration
- Custom residual CNN trained from scratch on single-channel inputs
- Normalization statistics estimated from a random subset of the training split
- Strong augmentation with flips, full-angle rotation, affine jitter, and blur
- Deeper classification head than the original baseline notebook
- `AdamW` with `OneCycleLR`, gradient clipping, and extended early stopping patience
- Test-time augmentation and micro-average ROC reporting in the final analysis

The code below follows the latest script configuration directly. Only the presentation is changed so the experiment can be rerun and read in notebook form.


In [ ]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    auc,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

plt.style.use("seaborn-v0_8-whitegrid")

# -- Config -------------------------------------------------------------------
SEED = 42
VAL_SIZE = 0.10
BATCH_SIZE = 256
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
EARLY_STOP_PATIENCE = 10
NUM_WORKERS = 4
WARMUP_PCT = 0.1
TTA_N = 8
NORM_SAMPLE_SIZE = 5_000

cwd = Path.cwd()
ROOT = cwd if (cwd / "dataset").exists() else cwd / "test1"
DATA_ROOT = ROOT / "dataset"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["no", "sphere", "vort"]
CLASS_LABELS = ["No substructure", "Subhalo substructure", "Vortex substructure"]
NUM_CLASSES = len(CLASS_NAMES)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

print(f"Device   : {device}")
print(f"Data root: {DATA_ROOT}")


## Data Loading And Split

Load the three class folders from `dataset/train` and `dataset/val`, create the stratified 90:10 internal validation split, and summarize how samples are distributed across train, validation, and holdout data. This keeps model selection on the internal split while preserving the task holdout set for supplemental reporting.


In [ ]:
def load_split(split_name: str):
    paths, labels = [], []
    for label, class_name in enumerate(CLASS_NAMES):
        class_dir = DATA_ROOT / split_name / class_name
        class_paths = sorted(class_dir.glob("*.npy"))
        if not class_paths:
            raise FileNotFoundError(f"No .npy files found in {class_dir}")
        paths.extend(class_paths)
        labels.extend([label] * len(class_paths))
    return paths, labels


full_train_paths, full_train_labels = load_split("train")
holdout_paths, holdout_labels = load_split("val")

train_paths, val_paths, train_labels, val_labels = train_test_split(
    full_train_paths,
    full_train_labels,
    test_size=VAL_SIZE,
    stratify=full_train_labels,
    random_state=SEED,
)

split_df = pd.DataFrame(
    {
        "split": (
            ["train"] * len(train_labels)
            + ["internal_val"] * len(val_labels)
            + ["holdout"] * len(holdout_labels)
        ),
        "label": train_labels + val_labels + holdout_labels,
    }
)
split_df["class_name"] = split_df["label"].map(lambda i: CLASS_NAMES[i])
print(
    split_df.groupby(["split", "class_name"]).size().unstack(fill_value=0).to_string()
)
print(f"\nTrain samples      : {len(train_paths):,}")
print(f"Internal val samples: {len(val_paths):,}")
print(f"Holdout samples    : {len(holdout_paths):,}")


## Normalization Statistics

Estimate scalar normalization statistics from a random subset of the training split. This matches the script behavior exactly and makes repeated runs much faster than scanning every training file while still keeping the normalization training-only.


In [ ]:
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(
    len(train_paths), size=min(NORM_SAMPLE_SIZE, len(train_paths)), replace=False
)
sample_arrays = np.stack(
    [np.load(train_paths[i]).astype(np.float32) for i in sample_indices]
)
mean = float(sample_arrays.mean())
std = float(sample_arrays.std())
std = max(std, 1e-6)

print(f"\nNorm stats (from {len(sample_indices):,} samples)")
print(f"  Mean: {mean:.6f}  Std: {std:.6f}")


## Dataset And DataLoaders

Define the dataset wrapper, the current augmentation policy, and the data loaders. Training uses flips, full-angle rotations, affine jitter, blur, and normalization, while validation and holdout apply only normalization so model selection reflects the raw evaluation setting.


In [ ]:
class LensDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = torch.from_numpy(np.load(self.paths[index]).astype(np.float32))
        if self.transform is not None:
            image = self.transform(image)
        return image, self.labels[index]


train_transform = transforms.Compose(
    [
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(180, interpolation=InterpolationMode.BILINEAR),
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.0)),
        transforms.Normalize([mean], [std]),
    ]
)

eval_transform = transforms.Normalize([mean], [std])

train_loader = DataLoader(
    LensDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    LensDataset(val_paths, val_labels, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
holdout_loader = DataLoader(
    LensDataset(holdout_paths, holdout_labels, eval_transform),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

sample_images, sample_labels = next(iter(train_loader))
print(f"\nBatch image shape : {tuple(sample_images.shape)}")
print(f"Batch labels shape: {tuple(sample_labels.shape)}")


## Model Definition

Use the custom residual CNN together with the deeper classification head from the current script. The backbone stays lightweight for 150x150 grayscale inputs, while the extra hidden layer in the head gives the classifier more capacity than the original single-layer projection.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.shortcut = (
            nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
            if stride != 1 or in_channels != out_channels
            else nn.Identity()
        )

    def forward(self, x):
        out = torch.nn.functional.silu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.nn.functional.silu(out + self.shortcut(x))


class LensResNet(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, 5, 2, 2, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
        )
        self.layer1 = nn.Sequential(ResidualBlock(32, 32), ResidualBlock(32, 32))
        self.layer2 = nn.Sequential(ResidualBlock(32, 64, 2), ResidualBlock(64, 64))
        self.layer3 = nn.Sequential(ResidualBlock(64, 128, 2), ResidualBlock(128, 128))
        self.layer4 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.SiLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x)
        return self.head(x)


model = LensResNet(num_classes=NUM_CLASSES).to(device)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")


## Optimization And Scheduler Setup

Configure the training objective, optimizer, and learning-rate schedule. This notebook keeps the current script choices unchanged: label-smoothed cross-entropy, `AdamW`, and a batch-stepped `OneCycleLR` schedule with a warmup phase.


In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LR,
    total_steps=total_steps,
    pct_start=WARMUP_PCT,
    anneal_strategy="cos",
)


## Evaluation Helpers

Define the standard evaluation path, the script's raw-space TTA implementation, and the split summarization helper. The TTA logic is left exactly as in `train.py`: each augmentation pass rebuilds a loader from raw images, applies the spatial transform, then normalizes before inference.


In [ ]:
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_labels, all_probs = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=pin_memory)
            labels = labels.to(device, non_blocking=pin_memory)
            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)

            total_loss += loss.item() * len(images)
            all_labels.append(labels.cpu())
            all_probs.append(probs.cpu())

    labels_np = torch.cat(all_labels).numpy()
    probs_np = torch.cat(all_probs).numpy()
    preds_np = probs_np.argmax(axis=1)

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float((preds_np == labels_np).mean()),
        "auc_macro": float(
            roc_auc_score(labels_np, probs_np, multi_class="ovr", average="macro")
        ),
        "labels": labels_np,
        "predictions": preds_np,
        "probabilities": probs_np,
    }


def evaluate_tta(model, paths, labels_list, criterion, n_aug=TTA_N):
    tta_full_transform = transforms.Compose(
        [
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(180, interpolation=InterpolationMode.BILINEAR),
            transforms.Normalize([mean], [std]),
        ]
    )

    model.eval()
    labels_np = np.array(labels_list)
    probs_sum = np.zeros((len(labels_list), NUM_CLASSES), dtype=np.float32)
    total_loss = 0.0

    for aug_idx in range(n_aug):
        loader = DataLoader(
            LensDataset(paths, labels_list, tta_full_transform),
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=pin_memory,
        )
        offset = 0
        with torch.no_grad():
            for images, labels in loader:
                images = images.to(device, non_blocking=pin_memory)
                labels_dev = labels.to(device, non_blocking=pin_memory)
                logits = model(images)
                if aug_idx == 0:
                    total_loss += criterion(logits, labels_dev).item() * len(images)
                probs = torch.softmax(logits, dim=1).cpu().numpy()
                probs_sum[offset : offset + len(probs)] += probs
                offset += len(probs)

    probs_np = probs_sum / n_aug
    preds_np = probs_np.argmax(axis=1)

    return {
        "loss": total_loss / len(labels_list),
        "accuracy": float((preds_np == labels_np).mean()),
        "auc_macro": float(
            roc_auc_score(labels_np, probs_np, multi_class="ovr", average="macro")
        ),
        "labels": labels_np,
        "predictions": preds_np,
        "probabilities": probs_np,
    }


def summarize_split(title, results):
    labels = results["labels"]
    predictions = results["predictions"]
    probabilities = results["probabilities"]
    labels_bin = label_binarize(labels, classes=list(range(NUM_CLASSES)))

    print("=" * 60)
    print(title)
    print("=" * 60)
    print(f"Loss      : {results['loss']:.4f}")
    print(f"Accuracy  : {results['accuracy']:.4f}")
    print(f"Macro AUC : {results['auc_macro']:.4f}")

    print("Per-class AUC:")
    for i, class_name in enumerate(CLASS_NAMES):
        class_auc = roc_auc_score(labels_bin[:, i], probabilities[:, i])
        print(f"  {class_name}: {class_auc:.4f}")

    print("Classification Report:")
    print(
        classification_report(
            labels, predictions, target_names=CLASS_NAMES, zero_division=0
        )
    )
    return labels, predictions, probabilities


## Training Loop

Train the model with `OneCycleLR`, gradient clipping, checkpointing, and the longer early-stopping patience from the script. Checkpoint selection still depends on internal validation macro AUC, matching the task emphasis on ROC/AUC rather than any single thresholded accuracy value.


In [ ]:
best_auc = -float("inf")
best_state = copy.deepcopy(model.state_dict())
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=pin_memory)
        labels = labels.to(device, non_blocking=pin_memory)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        train_loss += loss.item() * len(images)
        train_correct += (logits.argmax(dim=1) == labels).sum().item()
        train_total += len(images)

    val_metrics = evaluate(model, val_loader, criterion)
    current_lr = optimizer.param_groups[0]["lr"]

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss / train_total,
            "train_acc": train_correct / train_total,
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["accuracy"],
            "val_auc": val_metrics["auc_macro"],
            "lr": current_lr,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss/train_total:.4f} "
        f"train_acc={train_correct/train_total:.4f} | "
        f"val_loss={val_metrics['loss']:.4f} "
        f"val_acc={val_metrics['accuracy']:.4f} "
        f"val_auc={val_metrics['auc_macro']:.4f} | "
        f"lr={current_lr:.2e}"
    )

    if val_metrics["auc_macro"] > best_auc:
        best_auc = val_metrics["auc_macro"]
        best_state = copy.deepcopy(model.state_dict())
        torch.save(
            {
                "model": best_state,
                "history": history,
                "best_val_auc": best_auc,
                "seed": SEED,
            },
            ARTIFACTS / "best_model.pt",
        )
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP_PATIENCE:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_state)
print(f"\nBest internal val AUC: {best_auc:.4f}")


## Training Curves

Plot the tracked training loss together with validation accuracy, validation AUC, and the learning-rate schedule induced by `OneCycleLR`. These curves are useful for seeing whether the warmup and cosine decay behave as intended and where the best checkpoint falls along the trajectory.


In [ ]:
df = pd.DataFrame(history)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(df["epoch"], df["train_loss"], label="train")
ax1.plot(df["epoch"], df["val_loss"], label="internal val")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss curves")
ax1.legend()

ax2.plot(df["epoch"], df["train_acc"], label="train acc")
ax2.plot(df["epoch"], df["val_acc"], label="val acc")
ax2.plot(df["epoch"], df["val_auc"], label="val AUC")
ax2.set_xlabel("Epoch")
ax2.set_title("Accuracy & AUC")
ax2.legend()

plt.tight_layout()
plt.savefig(ARTIFACTS / "training_curves.png", dpi=150)
plt.show()


## Final Evaluation

Evaluate the selected checkpoint on the internal validation split with and without TTA, then report the script's holdout result using TTA. This mirrors the current `train.py` output exactly, including the distinction between standard validation metrics and the TTA-enhanced supplemental results.


In [ ]:
print("\nRunning standard evaluation on internal validation split ...")
val_results = evaluate(model, val_loader, criterion)
val_true, val_pred, val_prob = summarize_split(
    "INTERNAL VALIDATION RESULTS (standard)", val_results
)

print(f"\nRunning TTA evaluation (n={TTA_N}) on internal validation split ...")
val_tta_results = evaluate_tta(model, val_paths, val_labels, criterion, n_aug=TTA_N)
val_tta_true, val_tta_pred, val_tta_prob = summarize_split(
    "INTERNAL VALIDATION RESULTS (TTA)", val_tta_results
)

print("\nRunning TTA evaluation on holdout split ...")
holdout_tta_results = evaluate_tta(
    model, holdout_paths, holdout_labels, criterion, n_aug=TTA_N
)
y_true, y_pred, y_prob = summarize_split(
    "HOLDOUT RESULTS - TTA (SUPPLEMENTAL)", holdout_tta_results
)


## ROC Curves And Confusion Matrices

Plot the per-class ROC curves, the supplementary micro-average ROC curve, and confusion matrices for the TTA evaluation outputs. These figures match the current script behavior and provide a richer final summary than scalar metrics alone.


In [ ]:
def plot_results(title, labels, predictions, probabilities, save_path=None):
    labels_bin = label_binarize(labels, classes=[0, 1, 2])
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for i, class_name in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(labels_bin[:, i], probabilities[:, i])
        class_auc = roc_auc_score(labels_bin[:, i], probabilities[:, i])
        ax1.plot(fpr, tpr, label=f"{class_name} (AUC={class_auc:.3f})")

    fpr_micro, tpr_micro, _ = roc_curve(labels_bin.ravel(), probabilities.ravel())
    auc_micro = auc(fpr_micro, tpr_micro)
    ax1.plot(
        fpr_micro,
        tpr_micro,
        "k-",
        linewidth=2,
        label=f"micro-avg (AUC={auc_micro:.3f})",
    )

    ax1.plot([0, 1], [0, 1], "k--", alpha=0.4)
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title(f"{title}\nROC Curves")
    ax1.legend(fontsize=9)

    ConfusionMatrixDisplay.from_predictions(
        labels,
        predictions,
        display_labels=CLASS_NAMES,
        ax=ax2,
        cmap="Blues",
        colorbar=False,
    )
    ax2.set_title(f"{title}\nConfusion Matrix")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
    plt.show()


plot_results(
    "Internal Validation (TTA)",
    val_tta_true,
    val_tta_pred,
    val_tta_prob,
    save_path=ARTIFACTS / "roc_val_tta.png",
)
plot_results(
    "Holdout (TTA)", y_true, y_pred, y_prob, save_path=ARTIFACTS / "roc_holdout_tta.png"
)

print("\nAll done.  Artefacts saved to:", ARTIFACTS.resolve())
